# Symbolic & Numerical Matrix / Jacobian Calculator

General-purpose math workbench powered by `matrix_math_tool.py` (SymPy + NumPy).
Plug in **any** symbolic expressions, matrices, or numbers (degrees or radians) to compute:
1. **Analytic Jacobian** $\mathbf{J} = \frac{\partial \mathbf{f}}{\partial \mathbf{q}}$
2. **Matrix Inverse via Adjoint / Cofactor Method** ($\det$, $\mathbf{C}$, $\text{adj} = \mathbf{C}^T$, $\mathbf{M}^{-1}$)
3. **Linear System Solver** $\mathbf{A}\mathbf{x} = \mathbf{b}$ (exact inverse or Moore-Penrose pseudoinverse)
4. **Manipulability & Eigenvalues** ($\mathbf{A} = \mathbf{J}\mathbf{J}^T$, $w = \sqrt{\det(\mathbf{A})}$, semi-axes $\sigma_i = \sqrt{\lambda_i}$)

In [ ]:
import numpy as np
import sympy as sp
from matrix_math_tool import (
    eval_matrix,
    analytic_jacobian,
    invert_matrix_adjoint,
    solve_linear_system,
    manipulability_and_eigen,
)

np.set_printoptions(suppress=True, precision=4)
sp.init_printing(use_unicode=True)
print("[OK] Matrix Math Tool ready.")

## Tool 1: Analytic Jacobian ($\mathbf{J} = \frac{\partial \mathbf{f}}{\partial \mathbf{q}}$)
Edit `exprs`, `variables`, and `subs` (set `subs=None` for pure symbolic output).

In [ ]:
# Define any symbols you need
q1, q2, q3 = sp.symbols('q1 q2 q3', real=True)
L1, L2, L3 = sp.symbols('L1 L2 L3', real=True)

# Example: 3-DOF RRP position equations (replace with any 2D/3D equations!)
R = L2 + L3 + q3
exprs = [
    R * sp.cos(q1) * sp.cos(q2),
    R * sp.sin(q1) * sp.cos(q2),
    L1 + R * sp.sin(q2),
]
variables = [q1, q2, q3]

# Optional numerical values (angles listed in deg_vars are auto-converted from degrees -> radians)
subs = {q1: 0, q2: 30, q3: 0.30, L1: 0.50, L2: 0.10, L3: 0.40}
deg_vars = [q1, q2]

jac_res = analytic_jacobian(exprs, variables, subs=subs, deg_vars=deg_vars)

## Tool 2: Matrix Inverse via Adjoint / Cofactor Method
Works on **any** square symbolic or numerical matrix `M` ($2 \times 2$, $3 \times 3$, etc.).

In [ ]:
# Plug in any square matrix M (here we reuse J_sym from Tool 1, or define your own sp.Matrix)
M = jac_res["J_sym"]

inv_res = invert_matrix_adjoint(M, subs=subs, deg_vars=deg_vars)

## Tool 3: Solve Linear System ($\mathbf{A}\mathbf{x} = \mathbf{b}$) & Manipulability / Eigenvalues
Automatically uses exact inverse for square matrices and Moore-Penrose pseudoinverse for non-square matrices.

In [ ]:
# 1. Solve A * x = b (e.g., IVK: J_v * q_dot = v_cmd)
b_vec = [0, 0, -0.05]
sol_res = solve_linear_system(jac_res["J_sym"], b_vec, subs=subs, deg_vars=deg_vars)

print("\n" + "="*60 + "\n")

# 2. Manipulability & Eigenvalues of A = J * J^T
manip_res = manipulability_and_eigen(jac_res["J_sym"], subs=subs, deg_vars=deg_vars)